In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
% matplotlib inline
import onep

# Load the pickle file
with open('/Users/suthardr/Desktop/collection_fc_allmice.pkl', 'rb') as file:
    collection_fc = pickle.load(file)
# Define animals for each group
mouse_ids = ['astroF3', 'astroF5', 'astroF6', 'astroF7', 'astroF8', 'astroF9', 'astroF10',
             'astroM3', 'astroM4', 'astroM5', 'astroM6', 'astroM7', 'astroM8', 'astroM9', 'astroM10']
#Grab your mice and take accepted traces - convert to numpy arrays within mouse_id variable names
for mouse_id in mouse_ids:
    try:
        traces = collection_fc.animals[mouse_id].accepted_traces.to_numpy()
        globals()[mouse_id] = traces[:3303, :]
        print(f"Created variable {mouse_id} with shape {globals()[mouse_id].shape}")
    except Exception as e:
        print(f"Error loading {mouse_id}: {e}")

In [ ]:
def run_cross_validated_analysis_first_second_half(
    animal_id, traces, timestamps, save_path_prefix,
    do_shuffle=True, n_shuffles=10000, seed=None, two_sided=True
):
    # ---------------- helpers ----------------
    def shuffle_control_test(x, y, n_shuffles=1000, seed=None, two_sided=True):
        x = np.asarray(x); y = np.asarray(y)
        if x.shape[0] != y.shape[0]:
            raise ValueError(f"Length mismatch: x={x.shape[0]}, y={y.shape[0]}")
        if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
            raise ValueError("Non-finite values in x or y")
        rng = np.random.default_rng(seed)
        r_obs, p_obs = stats.spearmanr(x, y)
        sh = np.empty(n_shuffles, dtype=float)
        for i in range(n_shuffles):
            y_perm = rng.permutation(y)
            sh[i], _ = stats.spearmanr(x, y_perm)
        if two_sided:
            p_emp = (np.sum(np.abs(sh) >= abs(r_obs)) + 1) / (n_shuffles + 1)
        else:
            if r_obs >= 0:
                p_emp = (np.sum(sh >= r_obs) + 1) / (n_shuffles + 1)
            else:
                p_emp = (np.sum(sh <= r_obs) + 1) / (n_shuffles + 1)
        return {
            "r_obs": float(r_obs),
            "p_obs_parametric": float(p_obs),
            "shuffle_rhos": sh,
            "p_empirical": float(p_emp),
            "null_mean": float(np.mean(sh)),
            "null_std": float(np.std(sh, ddof=1)),
            "z_obs_vs_null": float((r_obs - np.mean(sh)) / (np.std(sh, ddof=1) + 1e-12)),
        }

    def plot_shuffle_null(shuffle_rhos, r_obs, figsize=(3.5, 3.5), dpi=600):
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        ax.hist(shuffle_rhos, bins=30, density=True)
        ax.axvline(r_obs, linestyle='--')
        ax.set_xlabel('Spearman ρ')
        ax.set_ylabel('Density')
        ax.set_title('Shuffle null (permute Y across cells)')
        return fig, ax

    # ---------------- analysis ----------------
    plt.rcParams.update({'font.size': 8, 'font.family': 'Arial'})

    # Compute ETA for shocks 1&3 and 2&4
    whole_eta_13, time_13 = onep.eta_individual_cells(traces.T, timestamps, [[120, 180]], window=28)
    whole_eta_24, time_24 = onep.eta_individual_cells(traces.T, timestamps, [[240, 300]], window=28)

    # Z-score starting at 140
    adjusted_whole_eta_1_3 = stats.zscore(whole_eta_13[:, 140:], axis=1)
    adjusted_whole_eta_2_4 = stats.zscore(whole_eta_24[:, 140:], axis=1)
    time_1_3 = time_13[140:]

    # Sort arrays by max
    sorted_array_1, idxs_13 = onep.maxsort(adjusted_whole_eta_1_3)
    sorted_array_2, idxs_24 = onep.maxsort(adjusted_whole_eta_2_4)

    # y-ticks for heatmaps
    first_last = [0, len(adjusted_whole_eta_1_3)]

    # --- Heatmaps ---
    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(7, 5), dpi=600)
    sns.set(style="ticks", font="Arial")
    sns.heatmap(adjusted_whole_eta_1_3[idxs_13], ax=ax1, cmap='mako', cbar=False,
                xticklabels=True, rasterized=True, yticklabels=True, vmin=-4, vmax=4)
    ax1.tick_params(axis='both', which='major', labelsize=8)
    ax1.set_xticks(np.linspace(0, len(time_1_3), 5))
    ax1.set_xticklabels(np.linspace(np.min(time_1_3), np.max(time_1_3), 5).astype(int), rotation=0)
    ax1.set_yticks(first_last)
    ax1.set_yticklabels([first_last[0], first_last[1]], rotation=0)
    ax1.set_title('First Half (1&2)', fontsize=10, pad=10)
    ax1.set_ylabel('Astrocyte #', fontsize=10)
    ax1.set_xlabel('Time (s)', fontsize=10)
    sns.despine(ax=ax1, left=True, bottom=True)

    sns.heatmap(adjusted_whole_eta_1_3[idxs_24], ax=ax2, cmap='mako', cbar=True,
                cbar_kws={'label': 'Z-score'}, xticklabels=True, rasterized=True, yticklabels=True, vmin=-4, vmax=4)
    ax2.tick_params(axis='both', which='major', labelsize=8)
    ax2.set_xticks(np.linspace(0, len(time_1_3), 5))
    ax2.set_xticklabels(np.linspace(np.min(time_1_3), np.max(time_1_3), 5).astype(int), rotation=0)
    ax2.set_yticks(first_last)
    ax2.set_yticklabels([first_last[0], first_last[1]], rotation=0)
    ax2.set_title('First Half Sorted by Second Half', fontsize=10, pad=10)
    ax2.set_xlabel('Time (s)', fontsize=10)
    sns.despine(ax=ax2, left=True, bottom=True)

    plt.tight_layout(pad=3)
    heatmap_fp = f"{save_path_prefix}_heatmaps.svg"
    fig.savefig(heatmap_fp, dpi=600, bbox_inches='tight')
    plt.close(fig)

    # --- Argmaxes ---
    max_idxs_1_3 = np.argmax(sorted_array_1, axis=1)
    argmax_1_3 = time_1_3[max_idxs_1_3]

    odd_sorted_even = adjusted_whole_eta_1_3[idxs_24]
    sorted_idx = np.argmax(odd_sorted_even, axis=1)
    argmax_1_3_sorted_2_4 = time_1_3[sorted_idx]

    # --- Spearman correlation ---
    rho_s, p_s = stats.spearmanr(argmax_1_3, argmax_1_3_sorted_2_4)
    print(f"{animal_id} - Spearman rho: {rho_s:.4f}, p-value: {p_s:.4e}")

    # --- Shuffle control (optional) ---
    shuffle_fp = None
    shuffle_empirical_p = None
    shuffle_null_mean = None
    shuffle_null_std = None
    z_obs_vs_null = None

    if do_shuffle:
        if seed is None:
            seed = hash(str(animal_id)) & 0xFFFFFFFF
        shres = shuffle_control_test(
            argmax_1_3, argmax_1_3_sorted_2_4,
            n_shuffles=n_shuffles, seed=seed, two_sided=two_sided
        )
        print(f"{animal_id} - shuffle null: mean={shres['null_mean']:.4f}, "
              f"sd={shres['null_std']:.4f}, empirical p={shres['p_empirical']:.4g}, "
              f"z={shres['z_obs_vs_null']:.2f}")
        fig_shuf, ax_shuf = plot_shuffle_null(shres["shuffle_rhos"], shres["r_obs"])
        shuffle_fp = f"{save_path_prefix}_shuffle_null.svg"
        fig_shuf.savefig(shuffle_fp, dpi=600, bbox_inches='tight')
        plt.close(fig_shuf)

        shuffle_empirical_p = shres["p_empirical"]
        shuffle_null_mean = shres["null_mean"]
        shuffle_null_std = shres["null_std"]
        z_obs_vs_null = shres["z_obs_vs_null"]

    # --- Regression data (clean) ---
    df = pd.DataFrame({
        'Argmax First sorted First': argmax_1_3,
        'Argmax First sorted Second': argmax_1_3_sorted_2_4
    })
    x_name = 'Argmax First sorted First'
    y_name = 'Argmax First sorted Second'
    _df = df[[x_name, y_name]].copy()
    _df = _df[np.isfinite(_df[x_name]) & np.isfinite(_df[y_name])]
    x = _df[x_name].to_numpy()
    y = _df[y_name].to_numpy()
    X = sm.add_constant(x)

    # --- Fit RLM ---
    model = sm.RLM(y, X).fit()

    # ---- R^2 variants ----
    y_pred = np.asarray(model.fittedvalues)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2_var = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # guard for degenerate correlation
    try:
        r2_corr = float(np.corrcoef(y, y_pred)[0, 1] ** 2)
    except Exception:
        r2_corr = np.nan

    rho_rlm = model.model.M.rho
    scale = getattr(model, "scale", 1.0) or 1.0
    resid_fit_scaled = (y - y_pred) / scale
    resid_null_scaled = (y - float(np.mean(y))) / scale
    loss_fit = float(np.sum(rho_rlm(resid_fit_scaled)))
    loss_null = float(np.sum(rho_rlm(resid_null_scaled)))
    r2_pseudo_robust = 1 - (loss_fit / loss_null) if loss_null > 0 else np.nan

    # ---- Extract flattened params ----
    beta  = np.asarray(model.params)        # [intercept, slope]
    pvals = np.asarray(model.pvalues)       # [p_intercept, p_slope]
    intercept = float(beta[0]) if beta.size > 0 else np.nan
    slope     = float(beta[1]) if beta.size > 1 else np.nan
    p_value_bias  = float(pvals[0]) if pvals.size > 0 else np.nan
    p_value_slope = float(pvals[1]) if pvals.size > 1 else np.nan

    # --- Plot: scatter + RLM line + 95% CI (bootstrap with fallback) ---
    fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=600)
    sns.set(style="ticks", font="Arial")
    sns.scatterplot(data=_df, x=x_name, y=y_name, color='#231f20', ax=ax)

    x_grid = np.linspace(x.min(), x.max(), 200)
    X_grid = np.column_stack([np.ones_like(x_grid), x_grid])
    y_hat = X_grid @ np.asarray(model.params)
    ax.plot(x_grid, y_hat, color='#231f20', lw=2, label='RLM fit')

    n_boot = 1000
    min_ok = 100
    rng = np.random.default_rng(12345)
    y_boot = np.full((n_boot, x_grid.size), np.nan, dtype=float)

    for b in range(n_boot):
        idx = rng.integers(0, len(_df), size=len(_df))
        xb = x[idx]; yb = y[idx]
        Xb = sm.add_constant(xb)
        try:
            mb = sm.RLM(yb, Xb).fit()
            y_boot[b, :] = X_grid @ np.asarray(mb.params)
        except Exception:
            pass

    counts = np.sum(np.isfinite(y_boot), axis=0)
    lower = np.full_like(y_hat, np.nan)
    upper = np.full_like(y_hat, np.nan)
    ok = counts >= min_ok
    if np.any(ok):
        lower[ok] = np.nanpercentile(y_boot[:, ok], 2.5, axis=0)
        upper[ok] = np.nanpercentile(y_boot[:, ok], 97.5, axis=0)

    if np.any(~ok):
        covb = np.asarray(model.cov_params())
        se_mean = np.sqrt(np.sum(X_grid @ covb * X_grid, axis=1))
        z = stats.norm.ppf(0.975)
        lower_analytic = y_hat - z * se_mean
        upper_analytic = y_hat + z * se_mean
        lower[~ok] = lower_analytic[~ok]
        upper[~ok] = upper_analytic[~ok]

    ax.fill_between(x_grid, lower, upper, alpha=0.2, label='95% CI')
    ax.set_xlabel('Argmax First Sorted First (s)', fontsize=10, fontname="Arial")
    ax.set_ylabel('Argmax First Sorted Second (s)', fontsize=10, fontname='Arial')
    ax.tick_params(labelsize=8)
    ax.set_ylim(0, 25)
    ax.set_xlim(0, 25)
    sns.despine(ax=ax)
    ax.legend(frameon=False)

    scatter_fp = f"{save_path_prefix}_scatter.svg"
    fig.savefig(scatter_fp, dpi=600, bbox_inches='tight')
    plt.close(fig)

    # ---- Return flattened summary ----
    return {
        "animal": animal_id,
        "spearman_rho": float(rho_s),
        "spearman_p": float(p_s),
        "slope": slope,
        "intercept": intercept,
        "p_value_slope": p_value_slope,
        "p_value_bias": p_value_bias,
        "r2_var": float(r2_var),
        "r2_corr": float(r2_corr) if np.isfinite(r2_corr) else np.nan,
        "r2_pseudo_robust": float(r2_pseudo_robust) if np.isfinite(r2_pseudo_robust) else np.nan,
        "heatmap_fp": heatmap_fp,
        "scatter_fp": scatter_fp,
        "shuffle_hist_fp": shuffle_fp,
        "shuffle_empirical_p": shuffle_empirical_p,
        "shuffle_null_mean": shuffle_null_mean,
        "shuffle_null_std": shuffle_null_std,
        "shuffle_z": z_obs_vs_null,
    }

In [ ]:
# ===================== batch runner =====================

trace_lookup = {mouse: globals()[mouse] for mouse in mouse_ids}

all_results = []
save_dir = '/Users/suthardr/Desktop/first_v_second'

for mouse in mouse_ids:
    print(f"Processing {mouse}...")
    traces = trace_lookup[mouse]
    timestamps = collection_fc.animals[mouse].Timestamps[:3303]
    res = run_cross_validated_analysis_first_second_half(
        animal_id=mouse,
        traces=traces,
        timestamps=timestamps,
        save_path_prefix=f"{save_dir}/FC_crossvalled_{mouse}"
    )
    all_results.append(res)

df_all_results = pd.DataFrame(all_results)

# put 'animal' first if present
if 'animal' in df_all_results.columns:
    cols = ['animal'] + [c for c in df_all_results.columns if c != 'animal']
    df_all_results = df_all_results[cols]

print(df_all_results)
df_all_results.to_csv(f'{save_dir}/crossval_summary_results.csv', index=False)
print(f"Saved: {save_dir}/crossval_summary_results.csv")

In [ ]:
from scipy.stats import ttest_rel

def plot_spearman_rho_paired(
    df,
    obs_col="spearman_rho",
    null_col="shuffle_null_mean",
    id_col="animal",
    title="",
    save_path=None
):
    """
    Draws side-by-side boxplots of observed Spearman rho vs shuffle-mean rho,
    overlays paired points per animal connected by line segments,
    and annotates a paired t-test.

    Expects df to have columns: [id_col, obs_col, null_col].
    """
    plt.rcParams.update({'font.size': 8, 'font.family': 'Arial', 'figure.figsize': (3.5, 3.5)})
    sns.set(style="ticks", font="Arial")

    # Keep rows with both values present
    d = df[[id_col, obs_col, null_col]].dropna().copy()
    if d.empty:
        raise ValueError("No rows with both observed and shuffle mean rhos present.")

    # Paired t-test
    tstat, pval = ttest_rel(d[obs_col], d[null_col])

    # Long form for plotting
    long = d.melt(
        id_vars=id_col,
        value_vars=[obs_col, null_col],
        var_name="condition",
        value_name="rho"
    )
    name_map = {obs_col: "Observed", null_col: "Shuffle mean"}
    long["condition"] = long["condition"].map(name_map)

    order = ["Observed", "Shuffle mean"]  # left→right
    palette = {"Observed": "#B39BC8", "Shuffle mean": "#C9D6DF"}  # keep your original purple

    fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=600)

    # Boxplots (background)
    sns.boxplot(
        data=long, x="condition", y="rho",
        order=order, palette=palette, ax=ax,
        width=0.6, showfliers=False
    )

    # Connect each animal’s pair and plot points
    x_pos = {cond: i for i, cond in enumerate(order)}
    for animal, grp in long.groupby(id_col):
        if set(order).issubset(set(grp["condition"])):
            y0 = grp.loc[grp["condition"] == "Shuffle mean", "rho"].values[0]
            y1 = grp.loc[grp["condition"] == "Observed", "rho"].values[0]
            ax.plot(
                [x_pos["Shuffle mean"], x_pos["Observed"]],
                [y0, y1],
                color="gray", alpha=0.6, linewidth=1, zorder=2
            )
            ax.scatter(
                [x_pos["Shuffle mean"], x_pos["Observed"]],
                [y0, y1],
                s=12, color="black", zorder=3
            )

    # Axis labels, limits, title
    ax.set_ylabel("Spearman's ρ", fontsize=10, fontname='Arial', labelpad=10)
    ax.set_xlabel("")
    ax.set_title(title, fontsize=10, pad=10, fontname='Arial')
    ax.set_ylim(0, 1)

    sns.despine(ax=ax)
    plt.tight_layout(pad=3)

    # Significance bar + text
    y_max = long["rho"].max()
    y_min = long["rho"].min()
    span = max(1e-6, y_max - y_min)
    y_bar = y_max + 0.05 * span
    ax.plot([x_pos["Shuffle mean"], x_pos["Observed"]], [y_bar, y_bar], color="black", linewidth=1)

    if save_path:
        fig.savefig(save_path, dpi=600, bbox_inches='tight', format='svg')

    return fig, ax, (tstat, pval)

fig, ax, (t, p) = plot_spearman_rho_paired(
    df_all_results,
    obs_col="spearman_rho",
    null_col="shuffle_null_mean",
    id_col="animal"
)

plt.savefig('/Users/suthardr/Desktop/first_v_second/FC_crossval_rho_paired_first_v_secondhalf.svg', dpi=600, bbox_inches='tight', format='svg')

In [ ]:
from scipy.stats import ttest_1samp

rhos = df_all_results['spearman_rho']
t_stat, p_val = ttest_1samp(rhos, 0)

print(f"One-sample t-test against zero:")
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")